# EEG · 05 · Final generative comparison (Experiment 5)
**Question:** does generation with the *correct* EEG beat permuted / zero?

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, get_device, load_json
from PIL import Image
from pathlib import Path
from src.features import load_clip
from src.evaluation import compute_generation_metrics
from src.generation import save_comparison_grid, case_grids
cfg = load_config('configs/EEG/exp05_generation_ablation.yaml')
device = get_device(cfg.get('runtime.device','auto'))
src_dir = Path('outputs') / cfg.get('generation.source_experiment')
params = load_json(src_dir/'metadata'/'generation_params.json')
ids = params['image_ids']
conditions = [c for c in ['correct','permuted','zero'] if (src_dir/'generated'/c).exists()]
load = lambda d: [Image.open(src_dir/'generated'/d/f'{i}.png').convert('RGB') for i in ids]
outputs = {'real': load('real'), 'image_ids': ids}
for c in conditions: outputs[c] = load(c)

In [ ]:
clip_bundle = load_clip(cfg, device)
import pandas as pd
rows, per_sample = [], {}
for c in conditions:
    r = compute_generation_metrics(outputs['real'], outputs[c], clip_bundle, device,
                                   use_ssim=cfg.get('generation.compute_ssim', False),
                                   use_lpips=cfg.get('generation.compute_lpips', False))
    per_sample[c] = r['per_sample']['clip_similarity']
    rows.append({'condition': c, **{k: r['metrics'].get(k) for k in ['mean_clip_similarity','mean_pixel_mse','mean_ssim','mean_lpips']}})
pd.DataFrame(rows)

## Comparison grid  [real | correct | permuted | zero]

In [ ]:
p = save_comparison_grid(outputs, str(Path(src_dir)/'grids'/'exp05_grid.png'),
                         column_order=('real',)+tuple(conditions), max_rows=6)
plt.figure(figsize=(12,10)); plt.imshow(Image.open(p)); plt.axis('off'); plt.show()

In [ ]:
if 'correct' in per_sample:
    saved = case_grids(outputs, per_sample['correct'], str(Path(src_dir)/'grids'),
                       column_order=('real',)+tuple(conditions), k=5)
    plt.figure(figsize=(12,8)); plt.imshow(Image.open(saved['best_cases'])); plt.axis('off'); plt.title('best cases'); plt.show()

**Conclusion:** the correct condition should show higher CLIP similarity to the real image than permuted/zero. If not, the generator is not making meaningful use of the EEG signal, and this must be stated in the report.